# Tutorial 19: Long-term Memory and Checkpointing in LangGraph

In this tutorial we explore LangGraph's two complementary memory systems: per-thread short-term memory managed by the checkpointer, and cross-thread long-term memory managed by the `InMemoryStore`. Together they let agents remember context within a session and across completely separate sessions.

## 1. Memory Systems Overview

| System | Scope | API | Use case |
|---|---|---|---|
| **Checkpointer** | Per thread | `MemorySaver`, `SqliteSaver` | Resume a conversation, HITL |
| **InMemoryStore** | Cross-thread | `store.put()`, `store.search()` | User preferences, long-term facts |

Think of the checkpointer as a session cache and the store as a database.

## 2. Setup

In [1]:
import os
import uuid
from typing import TypedDict, List, Annotated
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.memory import InMemoryStore
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from langchain_core.messages import trim_messages
import operator

llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0.1,
)

print("Setup complete.")

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Setup complete.


## 3. Short-term Memory with MemorySaver

The `MemorySaver` checkpointer persists the full graph state after every node. Reusing the same `thread_id` automatically resumes the conversation.

In [2]:
class ChatState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]


def chat_node(state: ChatState) -> ChatState:
    response = llm.invoke(state["messages"])
    return {"messages": [AIMessage(content=response.content)]}


chat_workflow = StateGraph(ChatState)
chat_workflow.add_node("chat", chat_node)
chat_workflow.set_entry_point("chat")
chat_workflow.add_edge("chat", END)

checkpointer = MemorySaver()
chat_app = chat_workflow.compile(checkpointer=checkpointer)

# Thread 1 — conversation A
config_a = {"configurable": {"thread_id": "user-alice-session"}}

r1 = chat_app.invoke({"messages": [HumanMessage(content="My name is Alice and I love hiking.")]}, config_a)
print("Turn 1:", r1["messages"][-1].content[:100])

r2 = chat_app.invoke({"messages": [HumanMessage(content="What do you know about me?")]}, config_a)
print("Turn 2:", r2["messages"][-1].content[:150])

Turn 1: Hello Alice. It's great to hear that you love hiking. There's something about being in nature, surro
Turn 2: I know that your name is Alice and you love hiking. That's about it. I don't have any other information about you, such as your age, location, or inte


## 4. Persistent Checkpointing with SqliteSaver

`SqliteSaver` persists state to a SQLite file, so conversations survive Python process restarts.

In [3]:
try:
    from langgraph.checkpoint.sqlite import SqliteSaver
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "Missing optional dependency 'langgraph-checkpoint-sqlite'. "
        "Install it with: pip install langgraph-checkpoint-sqlite"
    ) from e

# State is written to memory.db in the current directory
with SqliteSaver.from_conn_string(":memory:") as sqlite_checkpointer:
    persistent_app = chat_workflow.compile(checkpointer=sqlite_checkpointer)
    config_b = {"configurable": {"thread_id": "persistent-session-1"}}

    r = persistent_app.invoke({"messages": [HumanMessage(content="Remember: my favourite colour is blue.")]}, config_b)
    print("Persistent session response:", r["messages"][-1].content[:150])

    # In a real application you would re-open the same SqliteSaver file
    # and the graph would remember the conversation across process restarts
    print("\nState keys stored:", list(persistent_app.get_state(config_b).values.keys()))

Persistent session response: Blue is a beautiful colour. I'll keep that in mind for our conversation. Is there anything specific you'd like to talk about or would you like some bl

State keys stored: ['messages']


## 5. Long-term Memory with InMemoryStore

The `InMemoryStore` is a cross-thread key-value store. Agents can read and write memories that persist across different `thread_id` sessions.

**Namespaces** organise stored items, typically `("user_id", "category")`.

In [4]:
from langgraph.store.memory import InMemoryStore
from langgraph.config import get_store

store = InMemoryStore()


class MemoryAwareChatState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]
    user_id: str


def memory_chat_node(state: MemoryAwareChatState) -> MemoryAwareChatState:
    store = get_store()
    user_id = state["user_id"]
    namespace = ("users", user_id, "memories")

    # Retrieve stored memories for this user
    memories = store.search(namespace)
    memory_context = ""
    if memories:
        facts = [m.value.get("fact", "") for m in memories]
        memory_context = f"\n\nKnown facts about this user: {'; '.join(facts)}"

    # Build messages with memory context injected into system prompt
    system = f"You are a helpful assistant.{memory_context}"
    from langchain_core.messages import SystemMessage
    full_messages = [SystemMessage(content=system)] + state["messages"]
    response = llm.invoke(full_messages)

    # Extract and store any new user facts from the conversation
    last_user_msg = state["messages"][-1].content if state["messages"] else ""
    if any(kw in last_user_msg.lower() for kw in ["i am", "i like", "i love", "my name", "i prefer"]):
        store.put(namespace, str(uuid.uuid4()), {"fact": last_user_msg})

    return {"messages": [AIMessage(content=response.content)]}


memory_workflow = StateGraph(MemoryAwareChatState)
memory_workflow.add_node("chat", memory_chat_node)
memory_workflow.set_entry_point("chat")
memory_workflow.add_edge("chat", END)

# Inject both checkpointer (short-term) and store (long-term)
memory_app = memory_workflow.compile(checkpointer=MemorySaver(), store=store)
print("Memory-aware chat app compiled.")

Memory-aware chat app compiled.


In [5]:
user_id = "user-bob"

# Session 1 — Bob introduces himself
config_s1 = {"configurable": {"thread_id": "bob-session-1"}}
r1 = memory_app.invoke(
    {"messages": [HumanMessage(content="Hi! My name is Bob and I love Python programming.")], "user_id": user_id},
    config_s1
)
print("Session 1:", r1["messages"][-1].content[:150])

# Session 2 — completely different thread_id but same user_id
config_s2 = {"configurable": {"thread_id": "bob-session-2"}}
r2 = memory_app.invoke(
    {"messages": [HumanMessage(content="What do you know about me?")], "user_id": user_id},
    config_s2
)
print("\nSession 2 (new thread, same user):", r2["messages"][-1].content[:200])

Session 1: Nice to meet you, Bob. Python is a fantastic language for programming, and it's great to hear you enjoy working with it. What kind of projects or area

Session 2 (new thread, same user): I know that your name is Bob and you have a passion for Python programming. That's a great start! If you'd like to share more about yourself or your interests, I'm all ears (or rather, all text). I ca


In [6]:
# Inspect the store directly
stored = store.search(("users", user_id, "memories"))
print(f"\nStored memories for {user_id}:")
for item in stored:
    print(f"  - {item.value['fact']}")


Stored memories for user-bob:
  - Hi! My name is Bob and I love Python programming.


## 6. Context Window Management with trim_messages

As conversations grow, the message history may exceed the model's context window. `trim_messages()` keeps the most recent messages within a token budget.

In [7]:
from langchain_core.messages import trim_messages

# Simulate a long conversation history
long_history = [
    HumanMessage(content="Message 1: Tell me about Python."),
    AIMessage(content="Python is a versatile programming language."),
    HumanMessage(content="Message 2: What about LangChain?"),
    AIMessage(content="LangChain is a framework for building LLM applications."),
    HumanMessage(content="Message 3: And LangGraph?"),
    AIMessage(content="LangGraph extends LangChain for graph-based workflows."),
    HumanMessage(content="Message 4: What is the capital of France?"),
]

# Keep only messages that fit within 100 tokens, always preserving the last message
trimmed = trim_messages(
    long_history,
    max_tokens=100,
    strategy="last",
    token_counter=llm,
    include_system=True,
    allow_partial=False,
)

print(f"Original messages: {len(long_history)}")
print(f"Trimmed messages: {len(trimmed)}")
for msg in trimmed:
    print(f"  {msg.__class__.__name__}: {msg.content[:60]}")

/home/domenico/clone/langchain-langgraph-tutorial/.venv/lib/python3.12/site-packages/langchain_core/language_models/base.py:354: UserWarning: Using fallback GPT-2 tokenizer for token counting. Token counts may be inaccurate for non-GPT-2 models. For accurate counts, use a model-specific method if available.
  return len(self.get_token_ids(text))


Original messages: 7
Trimmed messages: 7
  HumanMessage: Message 1: Tell me about Python.
  AIMessage: Python is a versatile programming language.
  HumanMessage: Message 2: What about LangChain?
  AIMessage: LangChain is a framework for building LLM applications.
  HumanMessage: Message 3: And LangGraph?
  AIMessage: LangGraph extends LangChain for graph-based workflows.
  HumanMessage: Message 4: What is the capital of France?


## 7. Conclusion

In this tutorial we covered LangGraph's memory systems:
- `MemorySaver` for in-memory per-thread state persistence
- `SqliteSaver` for durable persistence across process restarts
- `InMemoryStore` for cross-thread, cross-session long-term memories
- Namespace-based organisation of stored facts
- `trim_messages()` to manage context window budgets

In Tutorial 20 we explore Time Travel — using the checkpoint history to replay, debug, and fork agent runs.